<a href="https://colab.research.google.com/github/MarkChernov123/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO

removed = df.duplicated().sum().copy()
print(removed)
df = df.drop_duplicates().copy()
log('drop duplicates', 'dropped', removed)
df.head()

15
[drop duplicates] dropped (15 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 2 — clean `price` -> float

In [4]:
# TODO
df['price'] = df['price'].str.strip('$').astype(float)
log('price', 'coerced to float', len(df))
df.head()

[price] coerced to float (300 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,7.5
3,3,Cheeseburger,Food,NaN,7.5
4,4,cheese burger,Apparel,1.0,7.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [11]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce').copy()

#removes nans
df = df.dropna().copy()
#print(df)
#only positives
df = df[df['qty'] >= 0].copy()
#print(df)

log('qty', 'coerced to numeric, dropped negative', len(df))
df.head()

[qty] coerced to numeric, dropped negative (275 row(s))


,order_id,item,category,qty,price,revenue
0,0,rain poncho,raingear,3.0,12.0,36.0
1,1,foam finger,apparel,1.0,7.5,7.5
2,2,cheeseburger,merch,1.0,7.5,7.5
4,4,cheeseburger,apparel,1.0,7.5,7.5
5,5,foam finger,food,3.0,6.0,18.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [12]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse

#print(df['item'].value_counts())

print('before:', sorted(df['item'].unique()))

df['item'] = df['item'].str.lower().str.strip().copy()
item_map = {'cheese burger': 'cheeseburger'}

df['item'] = df['item'].replace(item_map)
print('after: ', sorted(df['item'].unique()))
log('item', 'canonicalized', len(df))

before: ['cheeseburger', 'foam finger', 'rain poncho']
after:  ['cheeseburger', 'foam finger', 'rain poncho']
[item] canonicalized (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [13]:
# TODO

#print(df['category'].value_counts())
print('before:', sorted(df['category'].unique()))

df['category'] = df['category'].str.lower().str.strip().copy()
category_map = {'rain-gear': 'raingear'}

df['category'] = df['category'].replace(category_map)
print('after: ', sorted(df['category'].unique()))
log('category', 'canonicalized', len(df))


before: ['apparel', 'food', 'merch', 'raingear']
after:  ['apparel', 'food', 'merch', 'raingear']
[category] canonicalized (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [15]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isna().sum() == 0
assert df['category'].isna().sum() == 0
# TODO: assert something about item
assert df['item'].isna().sum() == 0
# TODO: assert something about category
assert df['category'].isna().sum()==0
print('clean:', df.shape)

clean: (275, 6)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [16]:
# TODO
df['revenue']= df['qty'] * df['price']
category_revenue= df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print("The category revennue is: ", category_revenue)
total_revenue=df['revenue'].sum()
print(f"The total revenue is : ${total_revenue:,.2f}")
#print(f"revenue      : ${orders['price'].sum():,.2f}")
#{category_revenue}")
#total_revenue = df['revenue'].sum()
#print(f"total revenue is: {total_revenue}


The category revennue is:  category
food        1656.0
raingear    1512.0
merch        856.5
apparel      715.5
Name: revenue, dtype: float64
The total revenue is : $4,740.00


**What I would tell the vendor:** _..._

I would tell the vendor to increase the stock for food as it generates the most revenue.

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_your answer here_

A)

Removing the non negative values for quanity affected the revenue total the most. The number before removing negative quantities was 4594.50 and after removing them the total was 4740.00.

B)

Someone else could have kept the rows with missing quanitity values and used a default value of one(1) for NaN values which would have increased the total revenue. I decided to drop them so that the report accurately reflects the correct revenue even if some of the products are dropped from the table.
